# HQDE CBT Multi-Dataset Benchmark - Kaggle 2xT4

This notebook runs the HQDE-style DeBERTa ensemble benchmark for thesis/paper comparison tables. It uses canonical 10-label cognitive-distortion mapping by default so datasets are compared against the same CBT taxonomy.

Target runtime:
- Kaggle GPU accelerator: **2x Tesla T4**
- HQDE ensemble workers: **4**
- vCPUs: **4**
- DataLoader subprocess workers: **0 by default** for Kaggle stability

Outputs:
- `benchmark_outputs/cbt_multi_dataset/cbt_multi_dataset_comparison.csv`
- `benchmark_outputs/cbt_multi_dataset/cbt_multi_dataset_comparison.json`
- `benchmark_outputs/cbt_multi_dataset/cbt_multi_dataset_comparison.md`
- per-dataset `classification_report.json`

Enable Kaggle internet before running because the notebook downloads the benchmark script, Hugging Face datasets, and DeBERTa weights.

In [ ]:
# Install HQDE package and benchmark dependencies.
# --no-deps keeps Kaggle's preinstalled CUDA/Torch stack intact.
!pip install -q --no-deps "hqde==0.1.12"
!pip install -q "transformers>=4.45.0" "datasets>=2.14.0" "accelerate>=0.20.0" "sentencepiece>=0.1.99"
!pip install -q "scikit-learn>=1.3.0" "pandas>=2.0.0" "tqdm>=4.65.0"

In [ ]:
import os
import sys
import platform
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1024**3:.1f} GB")
try:
    print(f"HQDE: {version('hqde')}")
except PackageNotFoundError:
    print("HQDE package not found. Re-run the install cell.")

if torch.cuda.device_count() < 2:
    print("WARNING: This notebook is configured for 2xT4. In Kaggle, select GPU T4 x2 accelerator.")

In [ ]:
# Download the benchmark runner from the current GitHub main branch.
# This keeps the notebook compact while using the tested benchmark implementation.
import subprocess
import urllib.request

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
SCRIPT_URL = "https://raw.githubusercontent.com/Prathmesh333/HQDE-PyPI/main/examples/cbt_multi_dataset_comparison.py"
SCRIPT_PATH = WORK_DIR / "cbt_multi_dataset_comparison.py"

urllib.request.urlretrieve(SCRIPT_URL, SCRIPT_PATH)
assert SCRIPT_PATH.exists(), "Benchmark script download failed"
print(f"Work directory: {WORK_DIR}")
print(f"Benchmark script ready: {SCRIPT_PATH}")
subprocess.run([sys.executable, str(SCRIPT_PATH), "--list-datasets"], check=True)


In [ ]:
# Kaggle 2xT4 / 4-worker benchmark configuration.
# HQDE_ENSEMBLE_WORKERS controls model ensemble workers, not PyTorch DataLoader subprocesses.
os.environ["HQDE_ENSEMBLE_WORKERS"] = "4"
os.environ["HQDE_DATALOADER_WORKERS"] = "0"
os.environ["HQDE_PIN_MEMORY"] = "0"
os.environ["HQDE_BATCH_SIZE"] = "8"
os.environ["HQDE_NUM_EPOCHS"] = "5"
os.environ["HQDE_MAX_LENGTH"] = "256"
os.environ["HQDE_MAX_TRAIN_SAMPLES"] = "1000"
os.environ["HQDE_MAX_EVAL_SAMPLES"] = "300"
os.environ["HQDE_OUTPUT_DIR"] = str(WORK_DIR / "benchmark_outputs" / "cbt_multi_dataset")
os.environ["HQDE_DATASETS"] = "danthareja,halil-gpt4,elliott-validation"
os.environ["HQDE_LABEL_MODE"] = "canonical10"
os.environ["HQDE_MODEL_NAME"] = "microsoft/deberta-v3-base"

print("Benchmark environment:")
for key in [
    "HQDE_ENSEMBLE_WORKERS",
    "HQDE_DATALOADER_WORKERS",
    "HQDE_PIN_MEMORY",
    "HQDE_BATCH_SIZE",
    "HQDE_NUM_EPOCHS",
    "HQDE_MAX_LENGTH",
    "HQDE_MAX_TRAIN_SAMPLES",
    "HQDE_MAX_EVAL_SAMPLES",
    "HQDE_OUTPUT_DIR",
    "HQDE_DATASETS",
    "HQDE_LABEL_MODE",
    "HQDE_MODEL_NAME",
]:
    print(f"  {key}={os.environ[key]}")

In [ ]:
# Dry run first: verifies dataset loading, split construction, exact-overlap checks, and table export.
# This does not download/train DeBERTa.
dry_run_cmd = [sys.executable, str(SCRIPT_PATH), "--quick-test", "--dry-run"]
print("Running:", " ".join(dry_run_cmd))
subprocess.run(dry_run_cmd, check=True)


In [ ]:
# Full 2xT4 run.
# Expected behavior: 4 HQDE ensemble workers mapped round-robin over cuda:0 and cuda:1.
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--datasets", os.environ["HQDE_DATASETS"],
    "--label-mode", os.environ["HQDE_LABEL_MODE"],
    "--epochs", os.environ["HQDE_NUM_EPOCHS"],
    "--ensemble-workers", os.environ["HQDE_ENSEMBLE_WORKERS"],
    "--batch-size", os.environ["HQDE_BATCH_SIZE"],
    "--max-length", os.environ["HQDE_MAX_LENGTH"],
    "--max-train-samples", os.environ["HQDE_MAX_TRAIN_SAMPLES"],
    "--max-eval-samples", os.environ["HQDE_MAX_EVAL_SAMPLES"],
    "--dataloader-workers", os.environ["HQDE_DATALOADER_WORKERS"],
    "--output-dir", os.environ["HQDE_OUTPUT_DIR"],
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Display the generated thesis/paper comparison table.
import pandas as pd
from IPython.display import Markdown, display

out_dir = Path(os.environ["HQDE_OUTPUT_DIR"])
csv_path = out_dir / "cbt_multi_dataset_comparison.csv"
md_path = out_dir / "cbt_multi_dataset_comparison.md"
json_path = out_dir / "cbt_multi_dataset_comparison.json"

if not csv_path.exists():
    raise FileNotFoundError(f"Missing output table: {csv_path}")

df = pd.read_csv(csv_path)
display(df)

if md_path.exists():
    display(Markdown(md_path.read_text(encoding="utf-8")))

print("Saved files:")
for path in [csv_path, json_path, md_path]:
    print(f"  {path} | exists={path.exists()}")

In [ ]:
# Package outputs for download from Kaggle.
import shutil

archive_base = WORK_DIR / "hqde_cbt_multi_dataset_outputs"
out_dir = Path(os.environ["HQDE_OUTPUT_DIR"])
if not out_dir.exists():
    raise FileNotFoundError(f"Missing benchmark output directory: {out_dir}")
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=out_dir.parent, base_dir=out_dir.name)
print(f"Created {archive_path}")


## Reporting Checklist

For thesis/paper reporting, record:
- Git commit hash from the HQDE repository
- Kaggle hardware: 2xT4, 4 vCPUs
- PyTorch, Transformers, Datasets, and HQDE versions
- Dataset rows/classes for each dataset
- Exact text overlap counts
- Accuracy, weighted F1, macro F1
- Runtime and peak CUDA memory from the table

Do not merge the dataset rows into one headline score. The datasets have different label spaces and collection methods.